In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

NUM_EPOCHS = 100

csv_path = '/playpen/mufan/levi/tianlong-chen-lab/material-super-resolution/ControlNet/__runs__/2024-11-20-bs-ds-4x4/log.csv'

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"The file {csv_path} does not exist. Please check the path.")

df = pd.read_csv(csv_path)
df = df[df['Epoch'] < NUM_EPOCHS]
df.head()

In [ ]:

required_columns = ['Step', 'Epoch', 'Train Loss', 'Val Loss', 'Val MSE', 'Val PSNR']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"The following required columns are missing from the CSV: {missing_columns}")
else:
    print("All required columns are present.")

# 4. Sort the Data
df_sorted = df.sort_values(by=['Epoch', 'Step']).reset_index(drop=True)
df_sorted.head()

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# 5. Compute Gaussian Smoothing
def compute_gaussian_smoothing(series, sigma):
    return gaussian_filter1d(series, sigma=sigma)

# 6. Set Parameters
sigma = 1000

# 7. Calculate Gaussian Smoothing for Losses
df_sorted['Train Loss Smoothed'] = compute_gaussian_smoothing(df_sorted['Train Loss'], sigma)
df_sorted['Val Loss Smoothed'] = compute_gaussian_smoothing(df_sorted['Val Loss'], sigma)

# 8. Plot Gaussian Smoothing for Training and Validation Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Smoothed Training Loss
axes[0].plot(df_sorted['Epoch'], df_sorted['Train Loss Smoothed'], color='blue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (Smoothed)')
axes[0].grid(True)

# Add sigma annotation to Training Loss plot
axes[0].text(0.95, 0.95, f'σ = {sigma}',
            transform=axes[0].transAxes,
            fontsize=12, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

# Plot Smoothed Validation Loss
axes[1].plot(df_sorted['Epoch'], df_sorted['Val Loss Smoothed'], color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss (Smoothed)')
axes[1].grid(True)

# Add sigma annotation to Validation Loss plot
axes[1].text(0.95, 0.95, f'σ = {sigma}',
            transform=axes[1].transAxes,
            fontsize=12, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# fix weird PSNR formatting
corrected_arr = []
for item in df_sorted['Val PSNR']:
    val = ""
    if "tensor" in item:
        val = item.split("tensor(")[1][:-1]
    else:
        val = item
    corrected_arr.append(float(val))
    
df_sorted['Val PSNR'] = corrected_arr

# 10. Apply Gaussian Smoothing to Validation Metrics
sigma_mse = 1000
sigma_psnr = 1000

df_sorted['Val MSE Smoothed'] = gaussian_filter1d(df_sorted['Val MSE'], sigma=sigma_mse)
df_sorted['Val PSNR Smoothed'] = gaussian_filter1d(df_sorted['Val PSNR'], sigma=sigma_psnr)

# 11. Plot Gaussian Smoothed Validation MSE and PSNR
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Smoothed Validation MSE
axes[0].plot(df_sorted['Epoch'], df_sorted['Val MSE Smoothed'], color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation MSE')
axes[0].set_title('↓ Validation Reconstruction MSE (Smoothed)')
axes[0].grid(True)

# Add sigma annotation to MSE plot
axes[0].text(0.95, 0.95, f'σ = {sigma_mse}',
             transform=axes[0].transAxes,
             fontsize=12, verticalalignment='top',
             horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

# Plot Smoothed Validation PSNR
axes[1].plot(df_sorted['Epoch'], df_sorted['Val PSNR Smoothed'], color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation PSNR')
axes[1].set_title('↑ Val Reconstruction PSNR (Smoothed)')
axes[1].grid(True)

# Add sigma annotation to PSNR plot
axes[1].text(0.95, 0.95, f'σ = {sigma_psnr}',
             transform=axes[1].transAxes,
             fontsize=12, verticalalignment='top',
             horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.tight_layout(); plt.show();